In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
#!git clone https://github.com/recsyspolimi/RecSys_Course_AT_PoliMi

os.chdir("../RecSys_Course_AT_PoliMi")

!pwd

#!python run_compile_all_cython.py

/Users/filippo/Documents/PoliMI/Recommender Systems/RecSys_Challenge_2025-26/RecSys_Course_AT_PoliMi


In [3]:
import os
import time 
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scipy.sparse as sps
import matplotlib.pyplot as pyplot
%matplotlib inline

from sklearn.model_selection import KFold
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from skopt.space import Real, Integer, Categorical
from Evaluation.Evaluator import EvaluatorHoldout
from HyperparameterTuning.SearchBayesianSkopt import SearchBayesianSkopt
from Recommenders.SLIM.SLIMElasticNetRecommender import SLIMElasticNetRecommender

Tensorflow is not available


In [4]:
df_train = pd.read_csv("data_train.csv")
df_test_user = pd.read_csv("data_target_users_test.csv")

In [5]:
def split_train_in_five_percentage_global_sample(URM_all, train_percentages):
    """
    The function splits an URM in five matrices based on provided percentages.
    :param URM_all: The full URM matrix
    :param train_percentages: A list of percentages (must sum to 1.0)
    :return: A list of 5 sparse matrices
    """

    import numpy as np
    from scipy.sparse import coo_matrix
    from Data_manager.IncrementalSparseMatrix import IncrementalSparseMatrix

    assert len(train_percentages) == 5, "You must provide exactly 5 percentages."
    assert abs(sum(train_percentages) - 1.0) < 1e-6, "Percentages must sum to 1.0."

    num_users, num_items = URM_all.shape

    # Builders for each of the 5 matrices
    builders = [
        IncrementalSparseMatrix(n_rows=num_users, n_cols=num_items, auto_create_col_mapper=False, auto_create_row_mapper=False)
        for _ in range(5)
    ]

    URM_all_coo = coo_matrix(URM_all)

    # Shuffle indices
    indices_for_sampling = np.arange(URM_all.nnz, dtype=np.int32)
    np.random.shuffle(indices_for_sampling)

    # Calculate the number of interactions for each split
    split_sizes = [int(URM_all.nnz * percentage) for percentage in train_percentages]
    cumulative_sizes = np.cumsum(split_sizes)

    # Divide the indices into 5 groups
    indices_splits = [
        indices_for_sampling[cumulative_sizes[i - 1]:cumulative_sizes[i]] if i > 0 else indices_for_sampling[:cumulative_sizes[i]]
        for i in range(5)
    ]

    # Populate the builders
    for i, builder in enumerate(builders):
        builder.add_data_lists(
            URM_all_coo.row[indices_splits[i]],
            URM_all_coo.col[indices_splits[i]],
            URM_all_coo.data[indices_splits[i]],
        )

    # Convert to sparse matrices
    sparse_matrices = [builder.get_SparseMatrix() for builder in builders]

    # Ensure all outputs are in csr_matrix format
    sparse_matrices = [sp.csr_matrix(matrix) for matrix in sparse_matrices]

    return sparse_matrices

In [6]:
from scipy.sparse import coo_matrix

#valore 1 per ogni coppia (row, col)
data = [1] * len(df_train)
df_train["row"] = df_train["row"].astype(int)
df_train["col"] = df_train["col"].astype(int)

# matrice COO
URM_all = sp.csr_matrix((data, (df_train["row"], df_train["col"])))

In [7]:
train_percentages = [0.2, 0.2, 0.2, 0.2, 0.2]  # Cinque parti uguali

URM_parts = split_train_in_five_percentage_global_sample(URM_all, train_percentages)
URM_parts

[<Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>]

In [ ]:
import time 

class SaveResults(object):
    
    def __init__(self):
        self.results_df = pd.DataFrame(columns=["result", "train_time (min)"])
    
    def __call__(self, optuna_study, optuna_trial):
        hyperparam_dict = optuna_trial.params.copy()
        hyperparam_dict["result"] = optuna_trial.values[0]
        
        # Retrieve the optimal number of epochs and training time from the "user attributes" of the trial
        #hyperparam_dict["epochs"] = optuna_trial.user_attrs["epochs"]
        hyperparam_dict["train_time (min)"] = optuna_trial.user_attrs["train_time (min)"]
        
        self.results_df.loc[len(self.results_df)] = hyperparam_dict
        
        
import gc
import time
import numpy as np
import scipy.sparse as sps

def objective_function_slim(optuna_trial):
    # 1. Suggerimento Iperparametri
    # alpha: penalità totale (L1+L2). Più è piccolo, più il modello è denso e lento (e mangia RAM).
    alpha = optuna_trial.suggest_float("alpha", 1e-5, 1e-2, log=True)
    
    # l1_ratio: 1.0 è Lasso (molto sparso/veloce), 1e-5 è quasi Ridge (molto denso/lento).
    l1_ratio = optuna_trial.suggest_float("l1_ratio", 1e-5, 1.0, log=True)
    
    # topK: limita il numero di pesi per riga. Cruciale per non far morire il PC.
    topK = optuna_trial.suggest_int("topK", 50, 1000)

    start_time = time.time()
    scores = []
    
    # 5-fold Cross Validation
    for i in range(len(URM_parts)):
        # Costruiamo il training set escludendo la fold di validazione corrente
        # Usiamo vstack per efficienza su matrici sparse
        URM_combined = sps.vstack([URM_parts[j] for j in range(len(URM_parts)) if j != i])
        URM_combined = sps.csr_matrix(URM_combined)

        # Inizializzazione e Fit
        recommender_instance = SLIMElasticNetRecommender(URM_combined)
        recommender_instance.fit(
            l1_ratio = l1_ratio,
            alpha = alpha,
            topK = topK,
            positive_only = True
        )
        
        # Valutazione sulla fold i-esima
        evaluator_test = EvaluatorHoldout(URM_parts[i], cutoff_list=[20])
        result, _ = evaluator_test.evaluateRecommender(recommender_instance)
        
        recall_val = result.loc[20, "RECALL"]
        scores.append(recall_val)
        
        # --- GESTIONE RAM ---
        # Eliminiamo gli oggetti pesanti e forziamo il Garbage Collector tra le fold
        del recommender_instance
        del URM_combined
        gc.collect()
        
    # Calcolo tempo e log
    duration_min = (time.time() - start_time) / 60
    optuna_trial.set_user_attr("train_time (min)", duration_min) 
    
    avg_recall = sum(scores) / len(scores)
    print(f"Trial {optuna_trial.number} - Avg Recall@20: {avg_recall:.5f} - Time: {duration_min:.1f} min")
    
    return avg_recall

In [9]:

import optuna
save_results = SaveResults()
optuna_study = optuna.create_study(direction="maximize")

optuna_study.optimize(
    objective_function_slim,
    callbacks=[save_results],
    n_trials = 500,
    n_jobs = 1  
)

[I 2025-12-31 17:54:17,460] A new study created in memory with name: no-name-cd1bb02a-ba5e-4bc2-a4b8-45a00998366a


SLIMElasticNetRecommender: URM Detected 140 ( 0.1%) users with no interactions.
SLIMElasticNetRecommender: Processed 4144 (59.5%) in 5.00 min. Items per second: 13.81
SLIMElasticNetRecommender: Processed 6969 (100.0%) in 8.31 min. Items per second: 13.98
EvaluatorHoldout: Ignoring 23 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27072 (100.0%) in 5.83 sec. Users per second: 4647
SLIMElasticNetRecommender: URM Detected 128 ( 0.1%) users with no interactions.
SLIMElasticNetRecommender: Processed 4192 (60.2%) in 5.00 min. Items per second: 13.97


[W 2025-12-31 18:09:47,189] Trial 0 failed with parameters: {'alpha': 5.121685092542084e-05, 'l1_ratio': 0.00933908970957277, 'topK': 507} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/optuna/study/_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/wp/jydg89697jzcwnllv2_yz6d40000gn/T/ipykernel_31490/3408937761.py", line 47, in objective_function_slim
    recommender_instance.fit(
    ~~~~~~~~~~~~~~~~~~~~~~~~^
        l1_ratio = l1_ratio,
        ^^^^^^^^^^^^^^^^^^^^
    ...<2 lines>...
        positive_only = True
        ^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/_testing.py", line 145, in wrapper
    return fn(*args, **kwargs)
  File "/Users/filippo/Documents/PoliMI/Recommender Systems/RecSys_Challenge_2025-26/RecSys_Course

KeyboardInterrupt: 

In [ ]:
# 1. Stampa i parametri migliori in formato dizionario (pronti da copiare)
print(f"Miglior Recall@20: {optuna_study.best_value:.6f}")
print("Migliori Iperparametri:")
print(optuna_study.best_params)

# 2. Se vuoi vedere il dettaglio completo del miglior trial
best_trial = optuna_study.best_trial
print(f"Trial numero: {best_trial.number}")
print(f"Parametri: {best_trial.params}")

In [ ]:
optuna_study.best_trial.params

In [ ]:
save_results.results_df

In [ ]:
from optuna.visualization import plot_parallel_coordinate
from optuna.visualization import plot_param_importances


plot_param_importances(optuna_study)

In [ ]:
#plot_parallel_coordinate(optuna_study, params=["topK", "shrink", "tversky_alpha", "tversky_beta"])

In [ ]:
best_index = save_results.results_df["result"].idxmax()
best_hyperparams = save_results.results_df.loc[best_index].to_dict()

del best_hyperparams["result"]
del best_hyperparams["train_time (min)"]
best_hyperparams

